<a href="https://colab.research.google.com/github/2403a54124-ux/NLP-NATURAL-LANGUAGE-PROCESS-/blob/main/Assignment_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 27.0 MB/s eta 0:00:00


In [2]:
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D
from tensorflow.keras.layers import GlobalMaxPooling1D, Dense, Concatenate
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from gensim.models import Word2Vec



In [3]:
texts = [
    "this movie is great",
    "i love this film",
    "this movie is terrible",
    "i hate this film",
    "excellent acting",
    "poor story"
]

labels = [1,1,0,0,1,0]


In [4]:
tokenizer = Tokenizer()

tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)

word_index = tokenizer.word_index

print(word_index)


{'this': 1, 'movie': 2, 'is': 3, 'i': 4, 'film': 5, 'great': 6, 'love': 7, 'terrible': 8, 'hate': 9, 'excellent': 10, 'acting': 11, 'poor': 12, 'story': 13}


In [5]:
print(sequences)


[[1, 2, 3, 6], [4, 7, 1, 5], [1, 2, 3, 8], [4, 9, 1, 5], [10, 11], [12, 13]]


In [6]:
max_length = 6

X = pad_sequences(sequences, maxlen=max_length)

y = np.array(labels)


In [7]:
X


array([[ 0,  0,  1,  2,  3,  6],
       [ 0,  0,  4,  7,  1,  5],
       [ 0,  0,  1,  2,  3,  8],
       [ 0,  0,  4,  9,  1,  5],
       [ 0,  0,  0,  0, 10, 11],
       [ 0,  0,  0,  0, 12, 13]], dtype=int32)

In [8]:
sentences = [text.split() for text in texts]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences,
    vector_size=50,
    window=3,
    min_count=1
)

In [9]:
vocab_size = len(word_index) + 1
embedding_dim = 50

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, index in word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[index] = w2v_model.wv[word]


In [10]:
embedding_matrix[1]


array([-1.07399793e-03,  4.73267166e-04,  1.02023119e-02,  1.80166923e-02,
       -1.86056253e-02, -1.42316381e-02,  1.29186967e-02,  1.79488957e-02,
       -1.00312913e-02, -7.52459047e-03,  1.47592099e-02, -3.06411344e-03,
       -9.07479599e-03,  1.31094847e-02, -9.72152408e-03, -3.63450008e-03,
        5.75223239e-03,  1.98787940e-03, -1.65689103e-02, -1.89018492e-02,
        1.46257766e-02,  1.01367170e-02,  1.35181239e-02,  1.53095310e-03,
        1.27009423e-02, -6.81132963e-03, -1.89375610e-03,  1.15414225e-02,
       -1.50482031e-02, -7.87208881e-03, -1.50221623e-02, -1.85761740e-03,
        1.90782268e-02, -1.46414209e-02, -4.66462551e-03, -3.87903419e-03,
        1.61555540e-02, -1.18599525e-02,  8.93461038e-05, -9.50883515e-03,
       -1.92088094e-02,  1.00123230e-02, -1.75231937e-02, -8.78071412e-03,
       -6.58443096e-05, -5.95010119e-04, -1.53203467e-02,  1.92282051e-02,
        9.96576715e-03,  1.84629504e-02])

In [11]:
input_layer = Input(shape=(max_length,))



In [13]:
embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False)(input_layer)

conv1 = Conv1D(filters=100, kernel_size=3, activation='relu')(embedding_layer)

conv2 = Conv1D(filters=100, kernel_size=4, activation='relu')(embedding_layer)

conv3 = Conv1D(filters=100, kernel_size=5, activation='relu')(embedding_layer)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
pool1 = GlobalMaxPooling1D()(conv1)
pool2 = GlobalMaxPooling1D()(conv2)
pool3 = GlobalMaxPooling1D()(conv3)


In [15]:
merged = Concatenate()([pool1, pool2, pool3])



In [16]:
dense = Dense(10, activation='relu')(merged)


In [17]:
output = Dense(1, activation='sigmoid')(dense)



In [18]:
model = Model(inputs=input_layer, outputs=output)



In [19]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [20]:
model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 6, 50)     │        700 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 4, 100)    │     15,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 3, 100)    │     20,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 2, 100)    │     25,100 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 100)       │          0 │ conv1d_3[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 300)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 10)        │      3,010 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         11 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 64,021 (250.08 KB)

 Trainable params: 63,321 (247.35 KB)

 Non-trainable params: 700 (2.73 KB)

In [21]:
model.fit(
    X,
    y,
    epochs=10,
    batch_size=2
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.1667 - loss: 0.6957  
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8333 - loss: 0.6911
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8333 - loss: 0.6879 
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8333 - loss: 0.6842
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8333 - loss: 0.6821 
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8333 - loss: 0.6798 
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8333 - loss: 0.6780
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8333 - loss: 0.6753 
Epoch 9/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 1.0000 - loss: 0.6712
Epoch 10/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 1.0000 - loss: 0.6684


In [22]:
test_text = ["this movie is amazing"]

seq = tokenizer.texts_to_sequences(test_text)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

print(prediction)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step
[[0.49966493]]
